# Pharmacophore

Generating a pharmacophore from any of the available methods
returns a `t2fpharm.pharm.Pharmacophore` object
that provides different methods and attributes
to analyze, process, and visualize the generated pharmacophore.

In [4]:
import t2fpharm
import t2fpharm_study  # To obtain a pre-generated pharmacophore

Here, we use pre-generated `Pharmacophore` objects for brevity:

In [17]:
manager = t2fpharm_study.manager()
PDB_ID = "1AQ1"
# Pharmacophore generated by the `largest_peaks` method
pharm_lp: t2fpharm.pharm.Pharmacophore = manager.job_pharmacophore(
    job_name="grid_search-lp-1",
    pdb_id=PDB_ID,
    job_idx=1721
)
# Pharmacophore generated from the receptor–ligand complex
pharm_complex: t2fpharm.pharm.Pharmacophore = manager.ligand_pharmacophore(PDB_ID)

## Features

Pharmacophore features are stored as a `pandas.DataFrame`
and can be accessed from the `features` property:

In [12]:
pharm_lp.features[:5]

,instance,type,label,center,radius,value
0,0,OA,"(19, 64, 49)","[-4.930499935150146, 33.254500961303705, 13.31...",1.1625,-0.98
1,0,OA,"(26, 57, 41)","[-2.8304999351501463, 31.154500961303707, 10.9...",1.1625,-0.923
2,0,OA,"(40, 57, 27)","[1.3695000648498539, 31.154500961303707, 6.714...",1.1625,-0.832
3,0,OA,"(50, 49, 28)","[4.369500064849854, 28.75450096130371, 7.01449...",1.1625,-0.733
4,0,OA,"(66, 36, 20)","[9.169500064849855, 24.854500961303707, 4.6144...",1.1625,-0.701


The DataFrame contains at least the following columns:
- `instance`: An identifier for the feature instance,
  e.g., for when the pharmacophore is derived
  from multiple receptors or ligands.
  If only a single instance is generated, this will be 0 for all features.
- `type`: A string representing the feature type,
   e.g., "hbond_donor", "hbond_acceptor", "hydrophobic", etc.
- `label`: An identifier for different features of the same type
   within the same instance. That is, for each unique (`instance`, `type`) pair,
   each feature has a unique label.
   Each feature in the whole pharmacophore can thus be uniquely identified
   by its (`instance`, `type`, `label`) triplet.
   When the pharmacophore is generated using the `largest_peaks` method,
   the labels correspond to the grid index of the selected points.
- `center`: A sequence of three real numbers representing
   the 3D coordinates of the feature.
- `radius`: A non-negative real number representing the feature radius.
   The radius represents the uncertainty of the feature center.
   When the pharmacophore is generated using the `largest_peaks` method,
   - if a filter function is used, the radius is equal to the filter radius for each type.
   - If no filter function is used, the radius is equal to the grid point radius
     (i.e., half of the grid spacing).
  
  When the pharmacophore is generated using the `cnn` method,
  the radius is equal radius of the cluster, as specified by the method's `radius_type` argument.
- `value`: Value of the corresponding field at `center`.
  This is only available for target-based pharmacophores.

When the pharmacophore is generated by a clustering algorithm,
additional columns are also available:
- `n_members`: Number of features in the cluster.
- `members`: Row indices of the original features that belong to this cluster.
- `center_<center_type>`: Center coordinates of the cluster
  according to each available `center_type`.
- `radius_<center_type>_<radius_type>`: Radius of the cluster
  according to each available `center_type` and `radius_type`.
- `value_<center_type>`: Value associated with the cluster center
  for each available `center_type`.

For more information, see the docstring of the `Pharmacophore.cluster()` method:

In [19]:
help(pharm_lp.cluster)

Help on method cluster in module t2fpharm.pharm:

cluster(function: Union[Callable[[numpy.ndarray, numpy.ndarray], t2fpharm.input.pharm.cluster.ClusteringResult], dict[str, Callable[[numpy.ndarray, numpy.ndarray], t2fpharm.input.pharm.cluster.ClusteringResult]]], weights: Union[pandas.core.series.Series, numpy.ndarray, Sequence[float], NoneType] = None, center_type: Union[Literal['average', 'mean', 'midpoint', 'function'], dict[str, Literal['average', 'mean', 'midpoint', 'function']]] = 'average', radius_type: Union[Literal['average', 'mean', 'max', 'min'], dict[str, Literal['average', 'mean', 'max', 'min']]] = 'average', per_instance: bool = True) -> Self method of t2fpharm.pharm.Pharmacophore instance
    Cluster pharmacophore features using provided clustering functions.

    The clustering is performed on center coordinates of each feature type;
    it can be used in one of two ways:
    - **Per instance**: To cluster features of the same type separately for each instance.
      Th

## Visualization

The pharmacophore can be visualized using the `display()` method.
The feature radii for visualization is taken from the `features` DataFrame,
unless `override_radius` is set to True:

In [14]:
pharm_lp.display(
    default_radius=1,
    overdide_radius=True
)

NGLWidget(gui_style='ngl')

## Matching

Any two pharmacophores can be matched against each other,
using the `match()` method of one of the pharmacophores:

In [20]:
pharm_lp.match(pharm_complex)

,instance,type,label,target_instance,target_label,radius_sum,distance,match
0,0,OA,1,0,"(40, 57, 27)",1.1625,1.256660,False
1,0,HD,1,0,"(36, 59, 26)",0.9000,1.082018,False
2,0,HD,2,0,"(45, 33, 42)",0.9000,1.568397,False
3,0,C,1,0,"(23, 42, 42)",1.2750,0.715092,True
4,0,C,2,0,"(37, 50, 28)",1.2750,0.541950,True
5,0,C,3,0,"(43, 45, 28)",1.2750,0.428797,True
6,0,C,4,0,"(22, 50, 41)",1.2750,1.241171,True
7,0,C,5,0,"(22, 50, 41)",1.2750,0.197080,True
8,0,C,6,0,"(37, 50, 28)",1.2750,1.388406,False
9,0,C,7,0,"(43, 45, 28)",1.2750,1.014916,True


In [21]:
pharm_complex.match(pharm_lp)

ValidationError: 1 validation error for PharmFeaturesInput
  Value error, Invalid feature types found: ['e+', 'e-']. Allowed: ['C', 'HD', 'OA'] [type=value_error, input_value={'features':     instance...pes': {'HD', 'C', 'OA'}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error